<a href="https://colab.research.google.com/github/VickyW2366/Shors-optimisations/blob/main/Optimization_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install qiskit
!pip install qiskit-aer
import qiskit
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.circuit.library import QFT
from qiskit.visualization import plot_histogram
from math import gcd
from fractions import Fraction
from sympy.ntheory import factorint
import random
from scipy.stats import norm
import numpy as np
import time

# Gate-Level Optimization for Modular Exponentiation
'''
The modular exponentiation step dominates circuit depth. We can optimize for specific N=15.
Current Simplified Approach:

Generic CX gates without optimization
qc.cx(0, n_count)
qc.cx(0, n_count+1)
qc.cx(1, n_count)
qc.cx(1, n_count+2)
qc.cx(2, n_count)
qc.cx(2, n_count+3)
'''

# Measures how long the program takes to run
start_time = time.monotonic()

# Optimized for N=15 with Known Structure:
def optimized_modular_exponentiation_15(qc, control_qubits, target_qubits, a):
    """
    Optimized modular exponentiation for N=15 using known structure.
    Reduces gate count by exploiting the group structure.

    For N=15, the multiplicative group modulo 15 has structure:
    - For a=7, order 4: 7 → 4 → 13 → 1
    - For a=2, order 4: 2 → 4 → 8 → 1
    - For a=4, order 2: 4 → 1
    """

    if a == 7:
        # Known permutation: |1⟩ → |7⟩ → |4⟩ → |13⟩ → |1⟩
        # Use SWAP-based implementation for efficiency
        # Control qubit 0 (LSB)
        qc.cswap(control_qubits[0], target_qubits[0], target_qubits[1])
        qc.cx(control_qubits[0], target_qubits[2])

        # Control qubit 1
        qc.cswap(control_qubits[1], target_qubits[1], target_qubits[2])
        qc.cx(control_qubits[1], target_qubits[0])

        # Control qubit 2 (MSB)
        qc.cswap(control_qubits[2], target_qubits[0], target_qubits[2])
        qc.cx(control_qubits[2], target_qubits[1])

    elif a == 4:
        # Order 2: |1⟩ → |4⟩ → |1⟩
        # Simpler mapping
        for control in control_qubits:
            qc.cswap(control, target_qubits[0], target_qubits[2])
            qc.cx(control, target_qubits[1])

    elif a == 2:
        # Order 4: |1⟩ → |2⟩ → |4⟩ → |8⟩ → |1⟩
        # Can be optimized using bit permutations
        qc.cx(control_qubits[0], target_qubits[0])
        qc.cx(control_qubits[1], target_qubits[1])
        qc.cx(control_qubits[2], target_qubits[2])

    print(f"Optimized modular exponentiation for a={a} using {len(control_qubits)*2} gates")

# Run the fuctuion to initialize a QuantumCircuit object
# Assuming 3 control qubits and 4 target qubits are needed for the example call
optimized_modular_exponentiation_15(QuantumCircuit(3 + 4), [0, 1, 2], [3, 4, 5, 6], 7)
print("Program took %s seconds to run" % (time.time() - start_time))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 84.6 MB/s eta 0:00:00
Optimized modular exponentiation for a=7 using 6 gates
Program took 1778484051.8814647 seconds to run
